#  DLinear → Kaggle Submission

ეს notebook იღებს **DLinear**-ს (საუკეთესო მოდელი აქამდე ტესტირებულებს შორის —
იხ. `model_experiment_DLinear.ipynb` და README) და აწარმოებს პროგნოზს Kaggle-ის
ოფიციალურ `test.csv`-ზე, `submission.csv`-ის ფორმირებით Kaggle-ზე ასატვირთად.

## რა განსხვავებაა `model_experiment_DLinear.ipynb`-სგან

| | `model_experiment_DLinear.ipynb` | `model_inference.ipynb` (ეს ფაილი) |
|---|---|---|
| მიზანი | მოდელის შეფასება/შედარება | რეალური submission-ის გენერაცია |
| Val | შიდა split (`train.csv`-ის ბოლო 13 კვირა) | — |
| სერიები | 271 (`MAX_SERIES=300`-დან) | **ყველა** `test.csv`-ში არსებული სერია |
| `horizon` | 13 კვირა | `test.csv`-ის სრული სიგრძე (~39 კვირა) |
| Output | WMAE მეტრიკა | `submission.csv` ფაილი |

**მნიშვნელოვანი:** `train.csv`-ის validation-ზე დათვლილი WMAE (1075.21 და ა.შ.)
**არასდროს გამოჩნდება** Kaggle-ის leaderboard-ზე — ეს ჩვენი შიდა შეფასებაა. საჯარო
score მხოლოდ ამ notebook-ის output-ის (`submission.csv`) ატვირთვის შემდეგ გამოჩნდება.


In [ ]:
!pip install wandb -q



In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### `data_prep.py`-ის ფუნქციები (inline)

In [ ]:
def merge_all(sales_df, features_df, stores_df):
    df = sales_df.merge(features_df, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
    df = df.merge(stores_df, on="Store", how="left")

    if "IsHoliday_feat" in df.columns:
        df = df.drop(columns=["IsHoliday_feat"])

    return df


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()

WANDB_PROJECT = "walmart-forecasting-statistical"


## 1. მონაცემების ჩატვირთვა

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

def load_raw_data_kaggle(data_dir=KAGGLE_DATA_DIR):
    train = pd.read_csv(f"{data_dir}/train.csv.zip")
    test = pd.read_csv(f"{data_dir}/test.csv.zip")
    features = pd.read_csv(f"{data_dir}/features.csv.zip")
    stores = pd.read_csv(f"{data_dir}/stores.csv")

    train["Date"] = pd.to_datetime(train["Date"])
    test["Date"] = pd.to_datetime(test["Date"])
    features["Date"] = pd.to_datetime(features["Date"])

    return train, test, features, stores


train, test, features, stores = load_raw_data_kaggle()
train_df = merge_all(train, features, stores)
train_df = train_df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
train_df["series_id"] = train_df["Store"].astype(str) + "_" + train_df["Dept"].astype(str)

test_df = test.merge(features, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
if "IsHoliday_feat" in test_df.columns:
    test_df = test_df.drop(columns=["IsHoliday_feat"])
test_df = test_df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_df["series_id"] = test_df["Store"].astype(str) + "_" + test_df["Dept"].astype(str)

test_dates = sorted(test_df["Date"].unique())
HORIZON = len(test_dates)
print(f"Test period: {test_dates[0]} -> {test_dates[-1]} ({HORIZON} კვირა)")
print(f"Train series: {train_df['series_id'].nunique()}, Test series: {test_df['series_id'].nunique()}")


## 2. DLinear არქიტექტურა

იგივე მოდელი, რაც `model_experiment_DLinear.ipynb`-შია — შედეგები პირდაპირ
შედარებადია. განსხვავება მხოლოდ `HORIZON`-შია (ახლა `test.csv`-ის სრული სიგრძე,
`model_experiment_DLinear.ipynb`-ის ტესტისეული 13 კვირის ნაცვლად).

In [ ]:
class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size=25):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg_pool = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        pad_left = (self.kernel_size - 1) // 2
        pad_right = self.kernel_size - 1 - pad_left
        front = x[:, :1].repeat(1, pad_left)
        end = x[:, -1:].repeat(1, pad_right)
        x_padded = torch.cat([front, x, end], dim=1)

        trend = self.avg_pool(x_padded.unsqueeze(1)).squeeze(1)
        seasonal = x - trend
        return trend, seasonal


class DLinear(nn.Module):
    def __init__(self, lookback, horizon, kernel_size=25):
        super().__init__()
        self.decomposition = SeriesDecomposition(kernel_size)
        self.linear_trend = nn.Linear(lookback, horizon)
        self.linear_seasonal = nn.Linear(lookback, horizon)

    def forward(self, x):
        trend, seasonal = self.decomposition(x)
        trend_out = self.linear_trend(trend)
        seasonal_out = self.linear_seasonal(seasonal)
        return trend_out + seasonal_out


## 3. Sliding windows — ტრენინგი **ყველა** სერიაზე

`model_experiment_DLinear.ipynb`-სგან განსხვავებით, აქ `MAX_SERIES` შეზღუდვას არ
ვაწესებთ — საბოლოო submission-ისთვის საჭიროა მოდელმა იცოდეს ყველა store-dept
სერიის შაბლონი, არა მხოლოდ 271-ის.

**Val split აქ არ გვჭირდება** — მთელი `train.csv` გამოიყენება ტრენინგისთვის,
რადგან საბოლოო prediction `test.csv`-ის მომავალ, უცნობ კვირებზეა (არა
retrospective validation).

In [ ]:
LOOKBACK = 52

def build_training_windows(raw_df, lookback=LOOKBACK, horizon=HORIZON):
    """ყველა სერიაზე sliding-window sample-ები — val split გარეშე """
    series_ids = raw_df["series_id"].unique()

    X_train, y_train, holiday_train = [], [], []

    for sid in series_ids:
        sub = raw_df[raw_df["series_id"] == sid].sort_values("Date")
        sales = sub["Weekly_Sales"].values.astype(np.float32)
        holidays = sub["IsHoliday"].values.astype(bool)

        n = len(sales)
        if n < lookback + horizon:
            continue

        for end in range(lookback, n - horizon + 1):
            X_train.append(sales[end - lookback:end])
            y_train.append(sales[end:end + horizon])
            holiday_train.append(holidays[end:end + horizon])

    return np.array(X_train), np.array(y_train), np.array(holiday_train)


X_train, y_train, h_train = build_training_windows(train_df)
print(f"Train windows: {X_train.shape} (ყველა სერიიდან, horizon={HORIZON})")


## 4. ტრენინგი

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y, is_holiday):
        self.X = X
        self.y = y
        self.is_holiday = is_holiday

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        h = self.is_holiday[idx].astype(np.float32)

        mean, std = x.mean(), x.std() + 1e-6
        x_norm = (x - mean) / std
        y_norm = (y - mean) / std

        return (
            torch.tensor(x_norm, dtype=torch.float32),
            torch.tensor(y_norm, dtype=torch.float32),
            torch.tensor(h, dtype=torch.float32),
        )


def weighted_mae_loss(y_pred, y_true, is_holiday, holiday_weight=5.0):
    weights = torch.where(is_holiday > 0.5, holiday_weight, 1.0)
    return (weights * torch.abs(y_pred - y_true)).sum() / weights.sum()


train_dataset = WindowDataset(X_train, y_train, h_train)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

KERNEL_SIZE = 25
EPOCHS = 15  # model_experiment_DLinear.ipynb-ში early stopping ~epoch 12-ზე გაჩერდა
LEARNING_RATE = 1e-3

model = DLinear(lookback=LOOKBACK, horizon=HORIZON, kernel_size=KERNEL_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

run = wandb.init(project=WANDB_PROJECT, name="DLinear_FullTrain_Inference", reinit=True)
wandb.config.update({
    "model_type": "DLinear",
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "kernel_size": KERNEL_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_windows": len(X_train),
    "purpose": "final_submission",
})

model.train()
for epoch in range(EPOCHS):
    epoch_losses = []
    for x, y, h in train_loader:
        x, y, h = x.to(device), y.to(device), h.to(device)

        optimizer.zero_grad()
        y_pred = model(x)
        loss = weighted_mae_loss(y_pred, y, h)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    train_loss = np.mean(epoch_losses)
    wandb.log({"epoch": epoch, "train_loss_norm": train_loss})
    if epoch % 3 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:3d} | train_loss (norm) = {train_loss:.4f}")


## 5. Model Registry — W&B Artifact-ად რეგისტრაცია

დავალება მოითხოვს საუკეთესო მოდელის Model Registry-ში შენახვას. MLflow-ის ნაცვლად
W&B-ს ვიყენებთ (იხ. README) — **W&B Artifacts** არის ამ ფუნქციონალის ეკვივალენტი:
ვერსირებული მოდელის შენახვა, საიდანაც პირდაპირ ატვირთვაც შესაძლებელია.

In [ ]:
MODEL_PATH = "/kaggle/working/dlinear_final.pt"
torch.save(model.state_dict(), MODEL_PATH)

artifact = wandb.Artifact(
    name="dlinear-walmart-forecasting",
    type="model",
    description="DLinear (shared weights, all series) — final model trained on full train.csv for Kaggle submission",
    metadata={
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "kernel_size": KERNEL_SIZE,
        "epochs": EPOCHS,
        "train_series": train_df["series_id"].nunique(),
    },
)
artifact.add_file(MODEL_PATH)
run.log_artifact(artifact)
print("მოდელი დარეგისტრირდა W&B Artifacts-ში (Model Registry ეკვივალენტი).")


## 6. Inference — პროგნოზი `test.csv`-ზე

თითოეული `test.csv`-ში არსებული `(Store, Dept)` წყვილისთვის:
- თუ საკმარისი ისტორია გვაქვს `train.csv`-ში (`>= LOOKBACK` კვირა) — ვიყენებთ
  ბოლო `LOOKBACK` კვირას DLinear-ის input-ად
- თუ არასაკმარისია ან სერია საერთოდ არ გვხვდება `train.csv`-ში — **fallback**:
  ვიყენებთ არსებული ისტორიის საშუალოს (ან, თუ საერთოდ არ არსებობს, გლობალურ
  საშუალო Weekly_Sales-ს) — ეს იშვიათი შემთხვევაა, მაგრამ საჭიროა submission-ის
  სისრულისთვის (Kaggle ყველა test row-ზე ითხოვს პროგნოზს)

In [ ]:
model.eval()

global_mean_sales = train_df["Weekly_Sales"].mean()
predictions = {}  # series_id -> np.array(horizon,) ან None

train_series_ids = set(train_df["series_id"].unique())
test_series_ids = test_df["series_id"].unique()

fallback_count = 0

with torch.no_grad():
    for sid in test_series_ids:
        sub = train_df[train_df["series_id"] == sid].sort_values("Date")
        sales = sub["Weekly_Sales"].values.astype(np.float32)

        if len(sales) >= LOOKBACK:
            window = sales[-LOOKBACK:]
            mean, std = window.mean(), window.std() + 1e-6
            x_norm = torch.tensor((window - mean) / std, dtype=torch.float32).unsqueeze(0).to(device)
            y_pred_norm = model(x_norm).cpu().numpy().flatten()
            y_pred = y_pred_norm * std + mean
        elif len(sales) > 0:
            # არასაკმარისი ისტორია DLinear-ისთვის -> fallback: ამ სერიის საშუალო, გამეორებული
            y_pred = np.full(HORIZON, sales.mean())
            fallback_count += 1
        else:
            # სერია საერთოდ არ არსებობს train.csv-ში -> გლობალური fallback
            y_pred = np.full(HORIZON, global_mean_sales)
            fallback_count += 1

        y_pred = np.clip(y_pred, 0, None)  # უარყოფითი გაყიდვები აზრს არ იძლევა
        predictions[sid] = y_pred

print(f"სულ {len(test_series_ids)} სერია, {fallback_count} მათგანზე fallback გამოყენებულია "
      f"({fallback_count / len(test_series_ids) * 100:.1f}%)")


## 7. Submission.csv-ის ფორმირება

Kaggle-ის მოთხოვნილი ფორმატი: `Id,Weekly_Sales`, სადაც `Id = Store_Dept_Date`.

In [ ]:
date_to_index = {d: i for i, d in enumerate(test_dates)}

submission_rows = []
for _, row in test_df.iterrows():
    sid = row["series_id"]
    date_idx = date_to_index[row["Date"]]
    pred_value = predictions[sid][date_idx]

    row_id = f"{row['Store']}_{row['Dept']}_{row['Date'].strftime('%Y-%m-%d')}"
    submission_rows.append((row_id, pred_value))

submission = pd.DataFrame(submission_rows, columns=["Id", "Weekly_Sales"])
print(submission.shape)
submission.head()


In [ ]:
SUBMISSION_PATH = "/kaggle/working/submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"submission.csv შენახულია: {SUBMISSION_PATH}")

wandb.log({"submission_rows": len(submission), "fallback_series_pct": fallback_count / len(test_series_ids) * 100})
run.finish()
